# Learner Activity: PyTorch NN -> RNN -> LSTM Forecasting

Use one daily operations dataset to compare three forecasting approaches:

- Dense neural network with fixed lag features
- RNN with rolling sequence windows
- LSTM with the same rolling sequence windows

This is a guided walkthrough and comparison lab. The goal is to understand the forecasting workflow and model tradeoffs, not to memorize PyTorch syntax.


## How To Use This Notebook

Look for the seven code cells labeled `TODO 1` through `TODO 7`. Those are the only required code blocks you need to fill in.

You do **not** need to memorize PyTorch syntax. Helper functions are provided so you can focus on the forecasting workflow:

1. Create a next-day target.
2. Build Dense NN lag features.
3. Scale training features and target values without leakage.
4. Complete the Dense NN layer sizes.
5. Choose the sequence window size.
6. Complete the RNN output layer.
7. Complete the LSTM output layer.

Important practical note: neural network regression models often train better when the target is scaled. In this notebook, models train on scaled demand values, then predictions are converted back to original demand-score units for metrics and plots.

Cells without a `TODO` banner are setup, helper, plotting, or evaluation cells. Run them, read the comments, and pause at the concept checkpoints. Do not edit helper cells unless your instructor asks you to experiment.

Each model section includes a 7-day forward forecast graph. The optional 14-day forecast section is an extension for scenario discussion, not required core work.


## Guided 3-Hour Lab Flow

Use this notebook as a coached lab rather than a solo coding quiz. Recommended pacing:

- 30-40 min: PyTorch basics, tensors, `nn.Module`, and the training loop intuition
- 25-30 min: Dense NN lag-feature baseline
- 35-45 min: RNN/LSTM sequence setup and shape walkthrough
- 30-40 min: Run, compare, interpret metrics, loss curves, and forward forecasts
- 20-30 min: Controlled experimentation
- 10 min: Reflection and wrap-up

Key concepts to actively discuss while running the notebook:

- Time-based validation split instead of random shuffling
- Target leakage and why `shift(-1)` creates a next-day target
- Feature scaling and target inverse-scaling
- Tensor shapes, especially sequence input shape `[batch, timesteps, features]`
- Why a more complex model does not automatically produce a better validation score


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

try:
    import torch
    from torch import nn
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for this activity. Install it with: pip install torch"
    ) from exc

from forecasting_activity_utils import (
    forecast_future_dense,
    forecast_future_sequence,
    make_loader,
    make_sequence_arrays,
    plot_forecast,
    plot_forward_forecast,
    plot_multiple_forward_forecasts,
    predict_model,
    rmse_mae,
)

DATA_FILE = Path("daily_operations_forecasting_data.csv")
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_FILE}")


## 1. Load And Inspect The Dataset

The dataset represents daily platform operations. The target we will forecast is `demand_score`.

In [2]:
df = pd.read_csv(DATA_FILE, parse_dates=["date"])
df.head()


,date,day_index,active_accounts,usage_hours,support_tickets,release_flag,incident_flag,promo_flag,demand_score
0,2025-01-01,0,1175,3430.5,86,1,0,0,516.70
1,2025-01-02,1,1191,3626.4,85,1,0,0,545.29
2,2025-01-03,2,1190,3689.5,85,1,0,0,551.42
3,2025-01-04,3,1194,3604.1,85,1,0,0,538.54
4,2025-01-05,4,1203,3318.2,81,0,0,0,518.19


In [3]:
print("Rows:", len(df))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
df.info()


Rows: 240
Date range: 2025-01-01 to 2025-08-28
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             240 non-null    datetime64[us]
 1   day_index        240 non-null    int64         
 2   active_accounts  240 non-null    int64         
 3   usage_hours      240 non-null    float64       
 4   support_tickets  240 non-null    int64         
 5   release_flag     240 non-null    int64         
 6   incident_flag    240 non-null    int64         
 7   promo_flag       240 non-null    int64         
 8   demand_score     240 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(6)
memory usage: 17.0 KB


## 2. Plot The Forecasting Target

Look for trend, weekly rhythm, spikes, or dips. This helps decide whether a simple fixed-window model is enough.

In [4]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["date"], y=df["demand_score"], mode="lines", name="Demand score"))
fig.update_layout(
    title="Daily Demand Score",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Demand score",
)
fig.show()


## 3. Create A Next-Day Forecast Target

We forecast tomorrow's `demand_score` from information available today and earlier.

Hint: avoid future leakage. The target should be shifted backward so today's row points to tomorrow's value.

### Understanding Check: Forecast Target

Answer in 2-3 short sentences before moving on:

1. What does one row represent after `next_day_demand_score` is created?
- After creating next_day_demand_score, each row contains the information about the current day together with the demand value for the next day as the target. This means that the model uses today's data in order to predict tomorrwo's demand.
2. Why does `shift(-1)` match a next-day forecast?
- 'shift(-1) match a next-day forecast so that the next day's demand aligns with the current row's features. This setup allows the model to learn how present conditions affect future demand.
3. What future information would cause leakage here?
- The future information that would cause leakage are the future values or future operational metrics. Leakage make the mode unreallistically accurate during training but unreliable in actual forecasting.


In [5]:
# TODO 1: Create the next-day forecasting target.
# Replace this None with -1 so today's row points to tomorrow's demand_score.
target_column = "demand_score"

model_df = df.copy()
model_df["next_day_demand_score"] = model_df[target_column].shift(-1)
model_df = model_df.dropna().reset_index(drop=True)

model_df[["date", "demand_score", "next_day_demand_score"]].head()


,date,demand_score,next_day_demand_score
0,2025-01-01,516.70,545.29
1,2025-01-02,545.29,551.42
2,2025-01-03,551.42,538.54
3,2025-01-04,538.54,518.19
4,2025-01-05,518.19,503.56


## 4. Helper Functions

The notebook keeps the two learning-critical helper groups visible here:

- normalization functions, because scaling training data without leakage is part of the forecasting workflow
- the PyTorch training loop, because it reinforces yesterday's neural-network pattern

Other reusable utilities, such as DataLoader creation, sequence-window creation, metrics, prediction helpers, and forecast plotting, are imported from forecasting_activity_utils.py so the notebook stays focused.


### Understanding Check: Scaling And Training

Answer in 1-2 short sentences before running the model sections:

1. Why should fit_standardizer use training data only?
- The fit_standardizer should use training data only so that exposing model validation or future information during preprocessing will be avoided. This helps maintain a fair and realistic evaluation if model performance.
2. What does inverse_standardizer do after model prediction?
- The inverse_standardizer converts the scaled prediction values back to their original range. This makes the forecast easier to interpret and compare with actual demand values.
3. What are the four basic actions inside a PyTorch training loop?
- The four basic actions inside a PyTorch training loop are making predictions, computing the loss, performing backpropagation, and updating model weights. These basic actions are repeated every epoch so that the model will be able to gradually improve predicitons.
4. Why is it helpful to move routine plotting and forecasting utilities out of the learner notebook?
- It is helpful to move routine plotting and forecasting utilities out of the learner notebook so that the notebook will be cleaner and also for easier understanding. It an also help improve code organization and allows the same functions to be reused in different experiments.

In [6]:
def time_split(df: pd.DataFrame, train_ratio: float = 0.8):
    """Split by time order. Do not shuffle time series rows."""
    split_index = int(len(df) * train_ratio)
    return df.iloc[:split_index].copy(), df.iloc[split_index:].copy()


def fit_standardizer(train_values: np.ndarray):
    """Return mean and std from training data only to avoid future leakage."""
    train_values = np.asarray(train_values, dtype=float)
    mean = train_values.mean(axis=0, keepdims=True)
    std = train_values.std(axis=0, keepdims=True)
    std = np.where(std == 0, 1, std)
    return mean, std


def apply_standardizer(values: np.ndarray, mean: np.ndarray, std: np.ndarray):
    return (np.asarray(values, dtype=float) - mean) / std


def inverse_standardizer(values: np.ndarray, mean: np.ndarray, std: np.ndarray):
    """Convert scaled predictions back to the original unit."""
    return (np.asarray(values, dtype=float) * np.asarray(std).reshape(-1)[0]) + np.asarray(mean).reshape(-1)[0]


def train_model(model, train_loader, val_loader, epochs=80, lr=0.01, grad_clip=1.0):
    """Reusable PyTorch training loop.

    Conceptual steps:
    1. Forward pass: model makes predictions.
    2. Loss: compare predictions to actual values.
    3. Backward pass: calculate gradients.
    4. Optional gradient clipping: prevent unstable recurrent updates.
    5. Optimizer step: update weights.
    """
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                predictions = model(X_batch)
                val_losses.append(loss_fn(predictions, y_batch).item())

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "val_loss": float(np.mean(val_losses)),
        })

    return pd.DataFrame(history)


### Imported Forward Forecast Helpers


These forward-forecast helpers are imported from forecasting_activity_utils.py. They create future scenario rows, recursively append each prediction, and draw the forward-forecast graphs used later in the notebook.


## 5. Dense NN: Build Fixed Lag Features

A dense NN cannot naturally read a sequence. We give it a fixed set of lag columns.

Hint: choose lag columns that only use current or past values.

In [7]:
# TODO 2: Build fixed lag features for the Dense NN baseline.
# Replace this None with 7 for one week of memory.
dense_df = model_df.copy()
lag_days = 14

# These loops are provided. They create past-value columns only, so they do not leak the future.
for lag in range(1, lag_days + 1):
    dense_df[f"demand_lag_{lag}"] = dense_df["demand_score"].shift(lag)
    dense_df[f"usage_lag_{lag}"] = dense_df["usage_hours"].shift(lag)

dense_df = dense_df.dropna().reset_index(drop=True)

# Fill this list with the lag columns plus same-day operational flags.
# Hint: include demand_lag_1..7, usage_lag_1..7, release_flag, incident_flag, promo_flag, support_tickets.
dense_feature_columns = [
    "demand_lag_1",
    "demand_lag_2",
    "demand_lag_3",
    "demand_lag_4",
    "demand_lag_5",
    "demand_lag_6",
    "demand_lag_7",
    
    "usage_lag_1",
    "usage_lag_2",
    "usage_lag_3",
    "usage_lag_4",
    "usage_lag_5",
    "usage_lag_6",
    "usage_lag_7",
    
    "release_flag",
    "incident_flag",
    "promo_flag",
    "support_tickets"]

dense_target_column = "next_day_demand_score"
dense_df[dense_feature_columns + [dense_target_column]].head()


,demand_lag_1,demand_lag_2,demand_lag_3,demand_lag_4,demand_lag_5,demand_lag_6,demand_lag_7,usage_lag_1,usage_lag_2,usage_lag_3,usage_lag_4,usage_lag_5,usage_lag_6,usage_lag_7,release_flag,incident_flag,promo_flag,support_tickets,next_day_demand_score
0,445.88,447.81,524.06,553.90,569.22,567.56,540.50,3149.6,3144.9,3307.2,3499.6,3568.7,3551.7,3407.9,0,0,0,77,545.37
1,532.20,445.88,447.81,524.06,553.90,569.22,567.56,3367.8,3149.6,3144.9,3307.2,3499.6,3568.7,3551.7,0,0,0,77,542.39
2,545.37,532.20,445.88,447.81,524.06,553.90,569.22,3508.7,3367.8,3149.6,3144.9,3307.2,3499.6,3568.7,0,0,0,81,535.61
3,542.39,545.37,532.20,445.88,447.81,524.06,553.90,3507.4,3508.7,3367.8,3149.6,3144.9,3307.2,3499.6,0,0,0,78,506.09
4,535.61,542.39,545.37,532.20,445.88,447.81,524.06,3405.0,3507.4,3508.7,3367.8,3149.6,3144.9,3307.2,0,0,0,84,487.72


In [8]:
# TODO 3: Scale Dense NN features and target using training data only.
train_dense_df, val_dense_df = time_split(dense_df, train_ratio=0.8)

X_train_dense_raw = train_dense_df[dense_feature_columns].to_numpy(dtype=float)
y_train_dense_raw = train_dense_df[dense_target_column].to_numpy(dtype=float)

X_val_dense_raw = val_dense_df[dense_feature_columns].to_numpy(dtype=float)
y_val_dense_raw = val_dense_df[dense_target_column].to_numpy(dtype=float)

# Replace this None with X_train_dense_raw.
dense_mean, dense_std = fit_standardizer(X_train_dense_raw)
X_train_dense = apply_standardizer(X_train_dense_raw, dense_mean, dense_std)
X_val_dense = apply_standardizer(X_val_dense_raw, dense_mean, dense_std)

# Replace this None with y_train_dense_raw.reshape(-1, 1).
# Neural network regression models often train better when the target is scaled.
dense_y_mean, dense_y_std = fit_standardizer(y_train_dense_raw)
y_train_dense = apply_standardizer(y_train_dense_raw.reshape(-1, 1), dense_y_mean, dense_y_std).reshape(-1, 1)
y_val_dense = apply_standardizer(y_val_dense_raw.reshape(-1, 1), dense_y_mean, dense_y_std).reshape(-1)

train_dense_loader = make_loader(X_train_dense, y_train_dense, batch_size=32, shuffle=True)
val_dense_loader = make_loader(X_val_dense, y_val_dense, batch_size=32, shuffle=False)


## 6. Train And Evaluate A Dense NN Baseline

Complete the model shape from the lecture:

`input_dim -> 32 -> 16 -> 1`

In [9]:
# TODO 4: Complete the Dense NN layer sizes.
# Target shape: input_dim -> 32 -> 16 -> 1
class DenseForecastNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),  # Replace None with 32.
            nn.ReLU(),
            nn.Linear(32, 16),       # Replace with 32, 16.
            nn.ReLU(),
            nn.Linear(16, 1),          # Replace None with 16.
        )

    def forward(self, x):
        return self.net(x)


dense_model = DenseForecastNN(input_dim=len(dense_feature_columns))
dense_history = train_model(dense_model, train_dense_loader, val_dense_loader, epochs=100, lr=0.03)
dense_history.tail()


,epoch,train_loss,val_loss
95,96,0.045484,0.510630
96,97,0.041763,0.424972
97,98,0.033034,0.445491
98,99,0.035383,0.511335
99,100,0.049347,0.512922


In [10]:
# Dense NN training/validation loss curve.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dense_history["epoch"],
    y=dense_history["train_loss"],
    mode="lines",
    name="Train loss",
))
fig.add_trace(go.Scatter(
    x=dense_history["epoch"],
    y=dense_history["val_loss"],
    mode="lines",
    name="Validation loss",
))
fig.update_layout(
    title="Dense NN: Training vs Validation Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE loss on scaled target",
    hovermode="x unified",
)
fig.show()


In [11]:
dense_actual_scaled, dense_pred_scaled = predict_model(dense_model, val_dense_loader)

# Convert scaled model outputs back to original demand_score units before scoring.
dense_actual = inverse_standardizer(dense_actual_scaled, dense_y_mean, dense_y_std)
dense_pred = inverse_standardizer(dense_pred_scaled, dense_y_mean, dense_y_std)

dense_metrics = rmse_mae(y_val_dense_raw, dense_pred)
dense_metrics


{'rmse': np.float64(19.543554164101206), 'mae': np.float64(13.807771231157794)}

In [12]:
plot_forecast(
    val_dense_df["date"],
    y_val_dense_raw,
    dense_pred,
    "Dense NN Validation Forecast: Fixed Lag Features",
)


### Dense NN Forward Forecast

This graph forecasts the next 7 days beyond the dataset. It uses recent history and simple future feature assumptions, so it is a planning scenario rather than a validation plot.

In [13]:
future_horizon = 7

dense_future_7 = forecast_future_dense(
    model=dense_model,
    history_df=df,
    feature_columns=dense_feature_columns,
    lag_days=lag_days,
    horizon=future_horizon,
    feature_mean=dense_mean,
    feature_std=dense_std,
    target_mean=dense_y_mean,
    target_std=dense_y_std,
)

plot_forward_forecast(
    df,
    dense_future_7,
    "Dense NN Forward Forecast: Next 7 Days",
)

dense_future_7


,date,forecast_demand_score,horizon_day
0,2025-08-29,657.389023,1
1,2025-08-30,619.300533,2
2,2025-08-31,587.184726,3
3,2025-09-01,547.115733,4
4,2025-09-02,566.913125,5
5,2025-09-03,594.287038,6
6,2025-09-04,618.077676,7


## 7. RNN Setup: Convert The Same Dataset Into Sequences

RNN and LSTM models use rolling windows directly.

Hint: sequence input shape is `[batch, timesteps, features]`.

### Understanding Check: Sequence Shape

Answer in 2-3 short sentences after creating the sequence arrays:

1. In `X_train_seq.shape`, what do samples, timesteps, and features mean?
- In 'X_train_seq.shape, the samples represent the total number of days or observaitons in each sequence, and features represent the variables used for prediction. Togteher, they define the structure of sequential input data.
2. Why can an RNN or LSTM use ordered windows more naturally than a Dense NN?
- RNN or LSTM are designed to process data in sequence order. They use ordered windows more naturally than a Dense NN as it allows them to capture temporal patterns more effectively. DNN treat inputs independently and they may not fully capture sequential realtionships.
3. What tradeoff changes when `window_size` gets larger?
- A larger window size allows the model to learn longer historical patterns, but it also increases computational cost and training time. Very large windows may also introduce unecessary information and make training hrder.

Sequence-window construction is handled by make_sequence_arrays from forecasting_activity_utils.py. It converts the same time-ordered data into arrays shaped [samples, timesteps, features] for the RNN and LSTM sections.


In [14]:
# TODO 5: Create sequence windows for RNN and LSTM.
sequence_feature_columns = [
    "demand_score",
    "active_accounts",
    "usage_hours",
    "support_tickets",
    "release_flag",
    "incident_flag",
    "promo_flag",
]

sequence_target_column = "next_day_demand_score"

# Replace this None with 14 for two weeks of history.
window_size = 21

sequence_df = model_df.copy()
train_sequence_source, val_sequence_source = time_split(sequence_df, train_ratio=0.8)
sequence_split_day_index = train_sequence_source["day_index"].max()

# Fit feature and target scalers on training rows only.
seq_mean, seq_std = fit_standardizer(train_sequence_source[sequence_feature_columns].to_numpy(dtype=float))

# Replace this None with train_sequence_source[sequence_target_column].to_numpy(dtype=float).reshape(-1, 1).
seq_y_mean, seq_y_std = fit_standardizer(train_sequence_source[sequence_target_column].to_numpy(dtype=float).reshape(-1, 1))

# Transform the full timeline with training-only scalers.
# This lets validation windows use prior training history without leaking validation target values.
sequence_scaled = sequence_df.copy()
sequence_scaled[sequence_feature_columns] = apply_standardizer(
    sequence_df[sequence_feature_columns].to_numpy(dtype=float),
    seq_mean,
    seq_std,
)
sequence_scaled[sequence_target_column] = apply_standardizer(
    sequence_df[sequence_target_column].to_numpy(dtype=float).reshape(-1, 1),
    seq_y_mean,
    seq_y_std,
).reshape(-1)

X_all_seq, y_all_seq, all_seq_dates, all_seq_day_indices = make_sequence_arrays(
    sequence_scaled, sequence_feature_columns, sequence_target_column, window_size
)

train_seq_mask = all_seq_day_indices <= sequence_split_day_index
val_seq_mask = all_seq_day_indices > sequence_split_day_index

X_train_seq = X_all_seq[train_seq_mask]
y_train_seq = y_all_seq[train_seq_mask]
train_seq_dates = all_seq_dates[train_seq_mask]

X_val_seq = X_all_seq[val_seq_mask]
y_val_seq = y_all_seq[val_seq_mask]
val_seq_dates = all_seq_dates[val_seq_mask]

y_val_seq_raw = inverse_standardizer(y_val_seq, seq_y_mean, seq_y_std)

print("Training sequence shape:", X_train_seq.shape)
print("Validation sequence shape:", X_val_seq.shape)

train_seq_loader = make_loader(X_train_seq, y_train_seq, batch_size=32, shuffle=True)
val_seq_loader = make_loader(X_val_seq, y_val_seq, batch_size=32, shuffle=False)


Training sequence shape: (169, 21, 7)
Validation sequence shape: (49, 21, 7)


## 8. Train And Evaluate An RNN

The RNN reads the window step by step and uses the final hidden state for the forecast.

In [15]:
# TODO 6: Complete the RNN output layer.
# The RNN hidden state has size hidden_size, and the forecast has one scaled numeric output.
class RNNForecast(nn.Module):
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, 1)  # Replace None with hidden_size.

    def forward(self, x):
        output, hidden = self.rnn(x)
        last_hidden = hidden[-1]
        return self.output_layer(last_hidden)


rnn_model = RNNForecast(input_size=len(sequence_feature_columns), hidden_size=32)
rnn_history = train_model(rnn_model, train_seq_loader, val_seq_loader, epochs=200, lr=0.003)
rnn_history.tail()


,epoch,train_loss,val_loss
195,196,0.003993,0.355280
196,197,0.005759,0.354789
197,198,0.004219,0.340763
198,199,0.004736,0.379543
199,200,0.005075,0.372014


In [16]:
# RNN training/validation loss curve.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rnn_history["epoch"],
    y=rnn_history["train_loss"],
    mode="lines",
    name="Train loss",
))
fig.add_trace(go.Scatter(
    x=rnn_history["epoch"],
    y=rnn_history["val_loss"],
    mode="lines",
    name="Validation loss",
))
fig.update_layout(
    title="RNN: Training vs Validation Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE loss on scaled target",
    hovermode="x unified",
)
fig.show()


In [17]:
rnn_actual_scaled, rnn_pred_scaled = predict_model(rnn_model, val_seq_loader)

# Convert scaled recurrent-model outputs back to original demand_score units.
rnn_actual = inverse_standardizer(rnn_actual_scaled, seq_y_mean, seq_y_std)
rnn_pred = inverse_standardizer(rnn_pred_scaled, seq_y_mean, seq_y_std)

rnn_metrics = rmse_mae(y_val_seq_raw, rnn_pred)
rnn_metrics


{'rmse': np.float64(21.313262543210186), 'mae': np.float64(15.646385353004456)}

In [18]:
plot_forecast(
    val_seq_dates,
    y_val_seq_raw,
    rnn_pred,
    "RNN Validation Forecast: Rolling Sequence Input",
)


### RNN Forward Forecast

The RNN uses the latest sequence window, predicts the next day, appends that prediction, then repeats until it reaches the 7-day horizon.

In [19]:
future_horizon = 7

rnn_future_7 = forecast_future_sequence(
    model=rnn_model,
    history_df=df,
    feature_columns=sequence_feature_columns,
    horizon=future_horizon,
    window_size=window_size,
    feature_mean=seq_mean,
    feature_std=seq_std,
    target_mean=seq_y_mean,
    target_std=seq_y_std,
)

plot_forward_forecast(
    df,
    rnn_future_7,
    "RNN Forward Forecast: Next 7 Days",
)

rnn_future_7


,date,forecast_demand_score,horizon_day
0,2025-08-29,609.252208,1
1,2025-08-30,552.661320,2
2,2025-08-31,532.752009,3
3,2025-09-01,579.267025,4
4,2025-09-02,603.278740,5
5,2025-09-03,630.285996,6
6,2025-09-04,635.020655,7


## 9. Train And Evaluate An LSTM

The LSTM uses the same sequence data as the RNN. The difference is the gated memory mechanism.

In [20]:
# TODO 7: Complete the LSTM output layer.
# The LSTM reuses the same scaled sequence data as the RNN.
class LSTMForecast(nn.Module):
    def __init__(self, input_size, num_layers=2, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_size, 1)  # Replace None with hidden_size.

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        last_hidden = hidden[-1]
        return self.output_layer(last_hidden)


lstm_model = LSTMForecast(input_size=len(sequence_feature_columns), hidden_size=32)
lstm_history = train_model(lstm_model, train_seq_loader, val_seq_loader, epochs=200, lr=0.003)
lstm_history.tail()


,epoch,train_loss,val_loss
195,196,0.003116,0.713799
196,197,0.002985,0.737557
197,198,0.002614,0.739511
198,199,0.002973,0.750189
199,200,0.002963,0.738911


In [21]:
# LSTM training/validation loss curve.
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lstm_history["epoch"],
    y=lstm_history["train_loss"],
    mode="lines",
    name="Train loss",
))
fig.add_trace(go.Scatter(
    x=lstm_history["epoch"],
    y=lstm_history["val_loss"],
    mode="lines",
    name="Validation loss",
))
fig.update_layout(
    title="LSTM: Training vs Validation Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE loss on scaled target",
    hovermode="x unified",
)
fig.show()


In [22]:
lstm_actual_scaled, lstm_pred_scaled = predict_model(lstm_model, val_seq_loader)

# Convert scaled recurrent-model outputs back to original demand_score units.
lstm_actual = inverse_standardizer(lstm_actual_scaled, seq_y_mean, seq_y_std)
lstm_pred = inverse_standardizer(lstm_pred_scaled, seq_y_mean, seq_y_std)

lstm_metrics = rmse_mae(y_val_seq_raw, lstm_pred)
lstm_metrics


{'rmse': np.float64(28.338601152901905), 'mae': np.float64(22.123242975298552)}

In [23]:
plot_forecast(
    val_seq_dates,
    y_val_seq_raw,
    lstm_pred,
    "LSTM Validation Forecast: Same Sequence Input With Gated Memory",
)


### LSTM Forward Forecast

The LSTM uses the same future scenario as the Dense NN and RNN. This keeps the comparison focused on model behavior, not different assumptions.

In [24]:
future_horizon = 7

lstm_future_7 = forecast_future_sequence(
    model=lstm_model,
    history_df=df,
    feature_columns=sequence_feature_columns,
    horizon=future_horizon,
    window_size=window_size,
    feature_mean=seq_mean,
    feature_std=seq_std,
    target_mean=seq_y_mean,
    target_std=seq_y_std,
)

plot_forward_forecast(
    df,
    lstm_future_7,
    "LSTM Forward Forecast: Next 7 Days",
)

lstm_future_7


,date,forecast_demand_score,horizon_day
0,2025-08-29,613.251985,1
1,2025-08-30,588.955418,2
2,2025-08-31,591.398914,3
3,2025-09-01,615.007139,4
4,2025-09-02,626.440924,5
5,2025-09-03,630.280783,6
6,2025-09-04,629.255844,7


## 10. Plot Training And Validation Loss Curves

Loss curves show whether each model kept improving, flattened out, or started to overfit. Because the target was scaled for training, these curves use scaled MSE loss; use the RMSE/MAE table in the next section for original-unit model comparison.


In [25]:
training_histories = {
    "Dense NN": dense_history,
    "RNN": rnn_history,
    "LSTM": lstm_history,
}

fig = go.Figure()
for model_name, history_df in training_histories.items():
    fig.add_trace(go.Scatter(
        x=history_df["epoch"],
        y=history_df["train_loss"],
        mode="lines",
        name=f"{model_name} train",
        line=dict(dash="solid"),
    ))
    fig.add_trace(go.Scatter(
        x=history_df["epoch"],
        y=history_df["val_loss"],
        mode="lines",
        name=f"{model_name} validation",
        line=dict(dash="dash"),
    ))

fig.update_layout(
    title="Training and Validation Loss Curves",
    xaxis_title="Epoch",
    yaxis_title="MSE loss on scaled target",
    hovermode="x unified",
    legend_title="Model / split",
)
fig.show()


### Understanding Check: Loss Curves

Answer in 2-3 short sentences after viewing the curves:

1. Which model had the lowest validation loss by the end of training?
- The RNN model had the lowest validation loss by the end of the training. This indicates that it was better at capturing long-term sequential patterns in the data.
2. Did any model show a large gap between training loss and validation loss?
- The model that show a large gap between training loss and validation loss is the RNN as it struggles remembering long-term patterns in sequential data. As training continued, it leanred the trainig data too specifically, which reduced training loss but resulted in weaker performance on unseen validation data.This suggests that this model may have started to overfit the training data.
3. Did the validation curve flatten before the final epoch?
- Yes, the validation curve flattened before the final epoch. This suggests that the model had already learned  most of the important patterns in data. Alsom it means that additional training provided only small improvements in validation performance.


## 11. Compare The Models

Lower RMSE and MAE are better. Metrics are reported in the original `demand_score` unit because predictions were converted back after training.

Do not choose the most complex model unless it improves validation performance enough to justify the complexity. It is normal for the Dense NN baseline to beat the RNN or LSTM on a small, stable dataset.


### Understanding Check: Model Choice

Answer in 2-3 short sentences after viewing the metrics:

1. Which model would you choose for this dataset, and what metric supports that choice?
- The model I would choose for this dataset is the LSTM model because it achieved better RMSE nad MAE results as compared to the other models. Lower error values indicate more accurate forecasting performance.
2. Did the RNN or LSTM improve enough to justify added complexity?
- The LSTM improve enough to justify added complexity for sequential data tasks. Its ability to remember long-term dependencies helped produce better predictions.
3. What would you inspect before trusting the 7-day forward forecast?
- I would inspect the validation metrics, prediction plots, and overall trend consistency before trusting the 7-day forward forecats. I would also verify if the assumptions used for future feature values are realistic.

## 12. Controlled Experimentation

After the default run, choose one or two experiments. Change only one thing at a time and rerun the affected section so the comparison is meaningful.

Recommended experiments:

- Change `lag_days` from `7` to `14`, then rebuild and retrain the Dense NN.
- Change `window_size` from `14` to `7` or `21`, then rebuild and retrain the RNN/LSTM inputs.
- Compare `hidden_size=16`, `32`, and `64` for the RNN or LSTM.
- In `make_future_feature_frame`, change future `promo_flag`, `release_flag`, or `incident_flag` assumptions.
- Compare the default 7-day forecast with the optional 14-day forecast below.

Experiment log:

| Experiment | What changed? | RMSE/MAE impact | Forecast shape impact | Keep or reject? |
|---|---|---|---|---|
| 1 | Change `lag_days` from `7` to `14`. | RMSE and MAE slightly improved. | The forecast became smoother and it also captures a longer trends better. | Keep  |
| 2 | Change `window_size` from `14` to `21`| RMSE and MAE improved because the model learned from a longer sequence history. | Forecast become more stable and less noisy with smoother trend behavior. | Keep |


In [26]:
comparison = pd.DataFrame([
    {"model": "Dense NN", **dense_metrics},
    {"model": "RNN", **rnn_metrics},
    {"model": "LSTM", **lstm_metrics},
])

comparison.sort_values("rmse")


,model,rmse,mae
0,Dense NN,19.543554,13.807771
1,RNN,21.313263,15.646385
2,LSTM,28.338601,22.123243


## 13. Extension: 14-Day Forward Forecast

The 7-day horizon is the classroom default. A 14-day horizon is useful for planning, but each extra day depends more heavily on earlier predictions and future feature assumptions. Treat this as an extension and discussion prompt after the required model comparison is complete.


In [27]:
optional_future_horizon = 14

dense_future_14 = forecast_future_dense(
    model=dense_model,
    history_df=df,
    feature_columns=dense_feature_columns,
    lag_days=lag_days,
    horizon=optional_future_horizon,
    feature_mean=dense_mean,
    feature_std=dense_std,
    target_mean=dense_y_mean,
    target_std=dense_y_std,
)

rnn_future_14 = forecast_future_sequence(
    model=rnn_model,
    history_df=df,
    feature_columns=sequence_feature_columns,
    horizon=optional_future_horizon,
    window_size=window_size,
    feature_mean=seq_mean,
    feature_std=seq_std,
    target_mean=seq_y_mean,
    target_std=seq_y_std,
)

lstm_future_14 = forecast_future_sequence(
    model=lstm_model,
    history_df=df,
    feature_columns=sequence_feature_columns,
    horizon=optional_future_horizon,
    window_size=window_size,
    feature_mean=seq_mean,
    feature_std=seq_std,
    target_mean=seq_y_mean,
    target_std=seq_y_std,
)

plot_multiple_forward_forecasts(
    df,
    {
        "Dense NN": dense_future_14,
        "RNN": rnn_future_14,
        "LSTM": lstm_future_14,
    },
    "Optional Forward Forecast Comparison: Next 14 Days",
)

pd.concat(
    [
        dense_future_14.assign(model="Dense NN"),
        rnn_future_14.assign(model="RNN"),
        lstm_future_14.assign(model="LSTM"),
    ],
    ignore_index=True,
)[["model", "horizon_day", "date", "forecast_demand_score"]]


,model,horizon_day,date,forecast_demand_score
0,Dense NN,1,2025-08-29,657.389023
1,Dense NN,2,2025-08-30,619.300533
2,Dense NN,3,2025-08-31,587.184726
3,Dense NN,4,2025-09-01,547.115733
4,Dense NN,5,2025-09-02,566.913125
5,Dense NN,6,2025-09-03,594.287038
6,Dense NN,7,2025-09-04,618.077676
7,Dense NN,8,2025-09-05,610.478594
8,Dense NN,9,2025-09-06,619.352059
9,Dense NN,10,2025-09-07,581.057494


## 14. Reflection

Answer each prompt in 2-3 short sentences. Use the validation metrics, forecast plots, and any experiment log entries as evidence.

1. Explain the complete forecasting setup: what information is available today, and what value is the model trying to predict?
- The model uses current and historical operational data such as demand  scores, usage hours, active accounts, and support tickets. The value that the model is trying to predict is the value of the next day's demand score based on these patterns.
2. When is the Dense NN baseline good enough for this kind of forecasting task?
- The Dense NN baseline is good enough for this kind of forecasting task when the relationships in the data are reltively enough for prediction. It is also useful when faster training and simpler implementation are preferred. It is also useful when faster training and simpler implementation are preferred.
3. What does the RNN gain by reading a sequence step by step instead of using fixed lag columns?
- The RNN can process information in chronological order, allowing it to capture changes and dependencies over time. This helps the model understand sequential behavior more naturally.
4. When would the LSTM be worth the extra complexity?
- The LSTM is worth using when long-term dependencies strongly influence future outcomes. Its memory mechanism helps retain important information across longer sequences.
5. Why is the 7-day forward forecast more reliable than the optional 14-day forecast?
- The 7-day forward forecasr is more reliable than the optional 14-day forecast because prediction errors accumulate over time. A 7-day forecast depends less on uncertain future assupmtions than a 14-day forecast.
6. Which future feature assumption would you change first for a promo, release, or incident scenario?
- I would change first the promo_flag, release_flag, and incident_flag values because these features directly affect user activity and demand behavior. Adjusting these feature assumptions would help model produce forecasts that better reflect the expected impact of promotions, software releases, or operational incidents.
